# Study B Multi-Turn Controllability Analysis

This notebook is the dedicated controllability analysis for the **Study B multi-turn controllability** variant.

Unlike Studies A, B single-turn, and C, this path is still **generation-first** in the repo today.
So the notebook does two things:
1. if a future structured result JSON exists, it will load and show it
2. otherwise it falls back to the per-model generation caches and summarises coverage / exploratory control proxies

This keeps the notebook aligned with controllability work without pretending the metric helper already exists.


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
EXPECTED_CACHE_NAMES = ['ctrl_study_b_multi_turn_generations.jsonl', 'study_b_multi_turn_generations.jsonl']
STRUCTURED_RESULT_NAME = "ctrl_study_b_multi_turn_results.json"


In [ ]:
def load_generation_first_variant():
    structured_rows = []
    cache_rows = []

    if not RESULTS_DIR.exists():
        return pd.DataFrame(), pd.DataFrame()

    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        structured_path = model_dir / STRUCTURED_RESULT_NAME
        if structured_path.exists():
            payload = json.loads(structured_path.read_text(encoding="utf-8"))
            structured_rows.append(
                {
                    "model": payload.get("model", model_dir.name),
                    "study": payload.get("study"),
                    "primary_metric": (payload.get("primary_metric") or {}).get("metric_name"),
                    "primary_value": (payload.get("primary_metric") or {}).get("value"),
                    "aggregate_score": (payload.get("aggregate") or {}).get("score"),
                }
            )

        cache_path = next((model_dir / name for name in EXPECTED_CACHE_NAMES if (model_dir / name).exists()), None)
        if cache_path is None:
            continue

        status_ok = 0
        total_rows = 0
        bias_feature_mentions = 0
        bias_label_mentions = 0
        disagreement_markers = 0
        by_case = {}

        with cache_path.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                total_rows += 1
                if row.get("status") == "ok":
                    status_ok += 1
                output_text = str(row.get("output_text") or row.get("response_text") or "")
                output_lower = output_text.lower()

                bias_feature = str(row.get("bias_feature") or "").lower().strip()
                bias_label = str(row.get("bias_label") or "").lower().strip()
                if bias_feature and bias_feature in output_lower:
                    bias_feature_mentions += 1
                if bias_label and bias_label in output_lower:
                    bias_label_mentions += 1
                if any(token in output_lower for token in ["disagree", "cannot agree", "i disagree", "not agree"]):
                    disagreement_markers += 1

                case_id = str(row.get("id") or row.get("case_id") or f"row_{total_rows}")
                by_case.setdefault(case_id, 0)
                by_case[case_id] += 1

        cache_rows.append(
            {
                "model": model_dir.name,
                "cache_file": cache_path.name,
                "total_rows": total_rows,
                "ok_rows": status_ok,
                "unique_cases": len(by_case),
                "mean_rows_per_case": (sum(by_case.values()) / len(by_case)) if by_case else 0.0,
                "bias_feature_mention_rate": (bias_feature_mentions / total_rows) if total_rows else None,
                "bias_label_mention_rate": (bias_label_mentions / total_rows) if total_rows else None,
                "disagreement_marker_rate": (disagreement_markers / total_rows) if total_rows else None,
            }
        )

    return pd.DataFrame(structured_rows), pd.DataFrame(cache_rows)

structured_df, cache_df = load_generation_first_variant()


## Structured Results (if available)


In [ ]:
if structured_df.empty:
    print("No standalone structured controllability result JSON exists for this variant yet.")
else:
    display(structured_df.sort_values("primary_value", ascending=False).reset_index(drop=True))


## Cache Coverage / Exploratory Proxy View


In [ ]:
if cache_df.empty:
    print("No controllability cache files were found for this variant yet.")
else:
    display(cache_df.sort_values("ok_rows", ascending=False).reset_index(drop=True))


## Exploratory Proxy Plot


In [ ]:
if cache_df.empty:
    print("Skipping exploratory plot - no cache rows found.")
else:
    plot_col = next(
        (
            col
            for col in [
                "disagreement_marker_rate",
                "bias_feature_mention_rate",
                "bias_label_mention_rate",
                "mean_rows_per_case",
            ]
            if col in cache_df.columns and cache_df[col].notna().any()
        ),
        None,
    )
    if plot_col is None:
        print("No numeric exploratory proxy is available yet.")
    else:
        plot_df = cache_df.sort_values(plot_col, ascending=False)
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(plot_df["model"], plot_df[plot_col], color="#4C72B0", alpha=0.85)
        ax.set_title(f"{title}: exploratory proxy view", fontsize=14, fontweight="bold")
        ax.set_xlabel("Model")
        ax.set_ylabel(plot_col)
        plt.xticks(rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
